# The Anatomy of AI · Lecture 10 · Curve Detectors — the notebook

In the lecture we met ten neurons in InceptionV1's layer **mixed3b** that each look for a curve bulging one way. Here you
rebuild every step yourself:

1. find where they fire on a car wheel;
2. find the photos they like best;
3. draw the test curves by hand and measure a **tuning curve**;
4. see the ten are one shape turned ten ways;
5. read the rule in their weights;
6. **write a curve detector by hand**, put it into the network in place of the real ones, and see how much still works.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab with a T4 GPU (Runtime → Change runtime type → T4), in about ten minutes. It downloads the
model (27 MB) and the lecture's own copy of the ImageNet validation photos (two files of about 350 MB), and uses only
the first 3,000 photos plus one car wheel.

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

### The photos
The lecture used ImageNet's 50,000 validation photos, 256 pixels square, cut to the middle 224 × 224 (the size
InceptionV1 takes). We read the same files, byte for byte.

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

### The network
InceptionV1 reads a photo in layers. A *layer* is a stack of maps; each map is one *neuron* (also called a unit or a
channel), slid across the whole picture, so it has a value at every place. We name a neuron by its layer and number:
**3b:379** is map 379 of layer mixed3b.

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

## 1. One layer, one name (Plate I)
The lecture's running example is test photo 45,840, a silver wheel on a red car. Show it to the network and keep two
things: the answer, and layer mixed3b.

In [ ]:
CURVES = [385, 343, 342, 388, 324, 340, 330, 349, 379, 406]   # the ten curve detectors of mixed3b
wheel_jpg, wheel_label = rows([45840])[45840]
wheel = decode(wheel_jpg)

with torch.no_grad(), Taps(["mixed3b", "softmax2_pre_activation_matmul"]) as T:
    net(prep(wheel[None].to(DEV)))
p = T["softmax2_pre_activation_matmul"][0, PERM].softmax(0)
top = p.topk(5)
print("InceptionV1 says:", [(NAMES[int(i)], round(float(v), 3)) for v, i in zip(top.values, top.indices)])
m3b = T["mixed3b"][0].cpu()
check("the photo's label", NAMES[wheel_label], "car wheel")
check("InceptionV1's first answer", NAMES[int(top.indices[0])], "car wheel")
check("P(car wheel)", float(top.values[0]), 1.0, tol=0.0005)
check("mixed3b: maps x height x width", tuple(m3b.shape), (480, 28, 28))

mixed3b has 480 neurons, each a 28 × 28 map: one value for every 8 × 8-pixel step across the picture. Where does each
of the ten curve detectors fire hardest on this wheel? The lecture found each at its own place round the rim.

In [ ]:
LECTURE_PEAK = {"385": 856.0, "343": 1034.0, "342": 764.5, "388": 833.5, "324": 633.5, "340": 724.5, "330": 938.0, "349": 1180.0, "379": 769.5, "406": 608.5}      # the course's census of all 50,000 photos (stored to half precision)
LECTURE_CELL = {"385": 213, "343": 443, "342": 526, "388": 662, "324": 686, "340": 683, "330": 566, "349": 424, "379": 205, "406": 181}     # the place, as row * 28 + column
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.imshow(wheel.permute(1, 2, 0).numpy()); ax.axis("off")
for c in CURVES:
    v, at = m3b[c].flatten().max(0)
    r, col = divmod(int(at), 28)
    px, py = 8 * col + 0.6, 8 * r - 0.1
    ax.plot(px, py, "o", ms=7, mfc=PAPER, mec=INK, mew=1.5)
    d = np.hypot(px - 112, py - 112) + 1e-6                     # label outward from the wheel's centre
    ax.text(px + 16 * (px - 112) / d, py + 16 * (py - 112) / d, f"3b:{c}", fontsize=8, color=INK, ha="center", va="center",
            bbox=dict(fc=PAPER, ec="none", pad=0.6))
    check(f"3b:{c}'s peak on the wheel", float(v), LECTURE_PEAK[str(c)], tol=0.006 * LECTURE_PEAK[str(c)])
    check(f"3b:{c}'s place", int(at), LECTURE_CELL[str(c)])
ax.set_title("Where each curve detector fires hardest", fontsize=11)
plt.show()

## 2. What does 3b:379 want? (Plate II)
The lecture searched all 50,000 photos. We search the first 3,000 and keep each photo's strongest value for every
mixed3b neuron. The census the course made gives the answer for these same 3,000 photos, so we can check ours.

In [ ]:
N = 3000
photos = rows(range(N))                    # {n: (jpeg, label)}
LABELS = np.array([photos[n][1] for n in range(N)])
best = torch.zeros(N, 480); where = torch.zeros(N, 480, dtype=torch.long)
t = time.time()
with torch.no_grad(), Taps(["mixed3b"]) as T:
    for s in range(0, N, 100):
        x = torch.stack([decode(photos[n][0]) for n in range(s, s + 100)]).to(DEV)
        net(prep(x))
        v, i = T["mixed3b"].flatten(2).max(2)
        best[s:s + 100], where[s:s + 100] = v.cpu(), i.cpu()
print(f"3,000 photos in {time.time() - t:.0f} s")
order = torch.argsort(best[:, 379], descending=True)[:12].tolist()
print("3b:379's twelve strongest photos:", [(n, NAMES[LABELS[n]], round(float(best[n, 379]))) for n in order])
check("3b:379's strongest photo of the first 3,000", order[0], 2052)
check("its value", float(best[order[0], 379]), 1152.0, tol=0.006 * 1152.0)
check("of the lecture's top 12 here, how many are in ours", len(set(order) & set([2052, 623, 2112, 1010, 677, 652, 1853, 700, 2665, 212, 633, 1680])), 12, tol=1)

In [ ]:
# Each patch is the 77 x 77 pixels the neuron can see from its strongest place (its receptive field).
fig, axs = plt.subplots(2, 6, figsize=(11, 4.2))
for a, n in zip(axs.flat, order):
    r, col = divmod(int(where[n, 379]), 28)
    cx, cy = 8 * col + 0.6, 8 * r - 0.1
    im = decode(photos[n][0]).permute(1, 2, 0).numpy()
    x0, y0 = int(np.clip(cx - 38, 0, 224 - 77)), int(np.clip(cy - 38, 0, 224 - 77))
    a.imshow(im[y0:y0 + 77, x0:x0 + 77]); a.axis("off")
    a.set_title(NAMES[LABELS[n]][:16], fontsize=8)
fig.suptitle("3b:379's twelve strongest patches, of the first 3,000 photos", fontsize=11)
plt.show()

Clocks, wheels, rims, domes: round, engineered things. Each patch holds a curve, bulging the same way.

## 3. A fair test (Plate IV)
Photos mix many things. To test what a neuron wants, show it one thing at a time: a thin dark curve on white, placed
exactly where the neuron in the middle of the map is looking.

**Where is it looking?** Start from a plain grey picture, ask how much the middle unit (row 14, column 14) of mixed3b's
5 × 5 branch would change if each pixel changed (the *gradient*), and take the centre of that blob.

In [ ]:
g = torch.full((1, 3, 224, 224), 0.5, device=DEV, requires_grad=True)
with Taps(["mixed3b_5x5_pre_relu_conv"]) as T:
    net(prep(g))
    T["mixed3b_5x5_pre_relu_conv"][0, :, 14, 14].abs().sum().backward()
G = g.grad[0].abs().sum(0).cpu().numpy()
yy, xx = np.mgrid[0:224, 0:224]
CX, CY = float((xx * G).sum() / G.sum()), float((yy * G).sum() / G.sum())
big = G > 0.01 * G.max()
print(f"centre ({CX:.2f}, {CY:.2f}); the 1%-of-peak box: x {xx[big].min()}..{xx[big].max()}, y {yy[big].min()}..{yy[big].max()}")
check("receptive-field centre, x", CX, 112.57, tol=0.01)
check("receptive-field centre, y", CY, 111.87, tol=0.01)

**The test curve.** An *arc* is part of a circle. Ours is drawn 2 px thick, at 4× size and shrunk (so its edges are
smooth). Its midpoint sits on the receptive-field centre, and it bulges toward an angle `phi`: 0 = right, 90 = up
(counter-clockwise, as in school geometry). Its length is 1.6 × the radius, at most 90 px, so it stays inside what the
neuron can see. A *bearing* is the same direction measured the map-reader's way, clockwise from up: bearing = 90 − phi.

In [ ]:
from PIL import ImageDraw

def arc(phi, radius, width=2.0, bg=1.0, ink=0.0, size=224, ss=4):
    a = math.radians(phi)
    S = size * ss
    im = Image.new("L", (S, S), int(round(bg * 255)))
    d = ImageDraw.Draw(im)
    cx, cy = CX * ss, CY * ss
    ox, oy = cx - radius * ss * math.cos(a), cy + radius * ss * math.sin(a)   # the circle's centre, behind the bulge
    half = (min(1.6 * radius, 90.0) / radius) / 2
    pts = [(ox + radius * ss * math.cos(a + t), oy - radius * ss * math.sin(a + t)) for t in np.linspace(-half, half, 200)]
    d.line(pts, fill=int(round(ink * 255)), width=max(1, int(round(width * ss))), joint="curve")
    im = im.resize((size, size), Image.LANCZOS)
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)[None].repeat(3, 1, 1)


def line(theta, length=60, width=2.0, size=224, ss=4):
    a = math.radians(theta)
    im = Image.new("L", (size * ss, size * ss), 255)
    dx, dy = math.cos(a) * length * ss / 2, -math.sin(a) * length * ss / 2
    cx, cy = CX * ss, CY * ss
    ImageDraw.Draw(im).line([(cx - dx, cy - dy), (cx + dx, cy + dy)], fill=0, width=max(1, int(round(width * ss))))
    im = im.resize((size, size), Image.LANCZOS)
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)[None].repeat(3, 1, 1)


bearing = lambda phi: int((90 - phi) % 360)
fig, axs = plt.subplots(1, 5, figsize=(10, 2.3))
for a, (phi, r) in zip(axs, [(90, 64), (0, 64), (-90 % 360, 64), (114, 32), (114, 160)]):
    a.imshow(arc(phi, r)[0], cmap="gray", vmin=0, vmax=1); a.axis("off")
    a.set_title(f"bearing {bearing(phi)}°, radius {r}", fontsize=9)
plt.show()

## 4. A tuning curve (Plate V)
Turn the arc all the way round, 2° at a time, and write down what 3b:379 says at the middle place each time. That list
is its **tuning curve**. Its *width* is the run of directions where it says more than half its best.

In [ ]:
ANGLES = list(range(0, 360, 2))
RADII = [16, 24, 32, 48, 64, 96, 160]

def respond(images, layers=("mixed3b",)):
    """Each layer's value at the middle place (row 14, column 14), for a list of pictures."""
    out = {L: [] for L in layers}
    with torch.no_grad(), Taps(list(layers)) as T:
        for s in range(0, len(images), 90):
            net(prep(torch.stack(images[s:s + 90]).to(DEV)))
            for L in layers:
                out[L].append(T[L][:, :, 14, 14].cpu())
    return {L: torch.cat(v).numpy() for L, v in out.items()}


def describe(t):
    """peak index, peak value, width in degrees (run above half the peak), and the value at the opposite direction."""
    pk = int(np.argmax(t)); w = 1
    for sgn in (1, -1):
        for d in range(1, 90):
            if t[(pk + sgn * d) % 180] > t[pk] / 2:
                w += 1
            else:
                break
    return pk, float(t[pk]), 2 * w, float(t[(pk + 90) % 180] / t[pk])


t379 = respond([arc(a, 64) for a in ANGLES])["mixed3b"][:, 379]
pk, peak, width, opp = describe(t379)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([bearing(a) for a in ANGLES], t379, ".", ms=3, color=INK)
ax.set_xlabel("bearing of the bulge (degrees, clockwise from up)"); ax.set_ylabel("3b:379, middle place")
ax.set_xticks(range(0, 361, 45)); plt.show()
check("3b:379's preferred bearing", bearing(ANGLES[pk]), 336)
check("its peak", peak, 956.0, tol=5.7)
check("its width (degrees)", width, 42)
check("at the opposite direction, as a share of the peak", opp, 0.0, tol=0.005)

## 5. Ten neurons, one shape (Plates VI–VIII)
Now all ten, at seven radii from tight (16 px) to gentle (160 px), plus straight lines. We also keep layer mixed3a,
the layer below, for later.

In [ ]:
t = time.time()
ARC = respond([arc(a, r) for r in RADII for a in ANGLES], ("mixed3a", "mixed3b"))
ARC3A = ARC["mixed3a"].reshape(len(RADII), len(ANGLES), -1)
ARC3B = ARC["mixed3b"].reshape(len(RADII), len(ANGLES), -1)
LIN = respond([line(a) for a in range(0, 180, 2)], ("mixed3a", "mixed3b"))
print(f"{len(RADII) * len(ANGLES) + 90} drawings in {time.time() - t:.0f} s")

ESSAY_ANGLE = {"385": 16, "343": 84, "342": 116, "388": 166, "324": 184, "340": 214, "330": 244, "349": 272, "379": 328, "406": 356}
ESSAY_WIDTH = {"385": 40, "343": 40, "342": 56, "388": 40, "324": 24, "340": 42, "330": 52, "349": 34, "379": 44, "406": 26}
LECTURE = {"bearing": {"385": 14, "343": 82, "342": 120, "388": 170, "324": 186, "340": 212, "330": 248, "349": 272, "379": 336, "406": 356}, "width": {"385": 38, "343": 38, "342": 50, "388": 34, "324": 22, "340": 38, "330": 46, "349": 36, "379": 42, "406": 22}, "radius": {"385": 64, "343": 48, "342": 64, "388": 96, "324": 64, "340": 64, "330": 64, "349": 48, "379": 64, "406": 64}, "peak": {"385": 832.2, "343": 794.7, "342": 366.0, "388": 657.4, "324": 715.3, "340": 872.9, "330": 780.7, "349": 1039.1, "379": 956.0, "406": 792.7}}
ours = {}
for c in CURVES:
    rb = int(np.argmax(ARC3B[:, :, c].max(1)))
    pk, peak, width, opp = describe(ARC3B[rb, :, c])
    ours[c] = dict(bearing=bearing(ANGLES[pk]), width=width, radius=RADII[rb], peak=peak, phi=ANGLES[pk])
print(" unit  bearing (essay)  width (essay)  best radius  peak")
for c in CURVES:
    o = ours[c]
    print(f" 3b:{c}  {o['bearing']:4d}° ({ESSAY_ANGLE[str(c)]:3d}°)     {o['width']:3d}° ({ESSAY_WIDTH[str(c)]:2d}°)      {o['radius']:4d} px   {o['peak']:7.1f}")
for c in CURVES:
    check(f"3b:{c}: bearing", ours[c]["bearing"], LECTURE["bearing"][str(c)])
    check(f"3b:{c}: width", ours[c]["width"], LECTURE["width"][str(c)])
    check(f"3b:{c}: best radius", ours[c]["radius"], LECTURE["radius"][str(c)])
diff = [abs(((ESSAY_ANGLE[str(c)] - ours[c]["bearing"] + 180) % 360) - 180) for c in CURVES]
check("largest difference from the essay's angle (degrees)", max(diff), 8)
check("mean difference from the essay's angle (degrees)", float(np.mean(diff)), 2.8, tol=0.05)

In [ ]:
# The ten tuning curves as petals round a compass: each points at its bearing.
fig = plt.figure(figsize=(5.4, 5.4)); ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_facecolor(PAPER)
b = np.radians([bearing(a) for a in ANGLES] + [bearing(ANGLES[0])])
r64 = RADII.index(64)
for c in CURVES:
    v = ARC3B[r64, :, c] / ARC3B[r64, :, c].max()
    ax.plot(b, np.r_[v, v[0]], color=INK, lw=1)
    ax.text(np.radians(ours[c]["bearing"]), 1.06, f"{c}", ha="center", va="center", fontsize=8,
            bbox=dict(fc=PAPER, ec="none", pad=0.5))
ax.set_yticks([]); ax.set_xticks(np.radians(range(0, 360, 45))); ax.set_xticklabels([])
ax.set_ylim(0, 1.12); ax.set_title("Ten neurons, one shape, ten directions (bearing clockwise from up)", fontsize=10, pad=12)
plt.show()

# Are they copies? Roll each curve so its peak sits in the same place, then compare them pairwise.
al = np.array([np.roll(ARC3B[r64, :, c], 90 - int(np.argmax(ARC3B[r64, :, c]))) / ARC3B[r64, :, c].max() for c in CURVES])
iu = np.triu_indices(10, 1)
C = np.corrcoef(al)[iu]
check("rolled to line up, mean pairwise correlation", C.mean(), 0.9156, tol=0.002)
check("the least alike pair", C.min(), 0.7713, tol=0.004)

## 6. Turn the photo (Plate IX)
If the ten are one detector turned ten ways, then turning a photo a quarter turn should hand each one's job to the
detector a quarter turn round. Take photos 1,000–2,999 and the same photos turned. For each detector on the turned
photos, find which detector on the originals it agrees with most (correlation of each photo's peak).

In [ ]:
def peaks_turned(k):
    out = []
    with torch.no_grad(), Taps(["mixed3b"]) as T:
        for s in range(1000, 3000, 100):
            x = torch.stack([decode(photos[n][0]) for n in range(s, s + 100)]).to(DEV)
            net(prep(torch.rot90(x, k, dims=(2, 3))))
            out.append(T["mixed3b"][:, CURVES].amax((2, 3)).cpu())
    return torch.cat(out).numpy()

P0, P1 = peaks_turned(0), peaks_turned(1)
Cm = np.corrcoef(P1.T, P0.T)[:10, 10:]
moves = [(CURVES[i], CURVES[int(np.argmax(Cm[i]))]) for i in range(10)]
shift = [(ours[b]["bearing"] - ours[a]["bearing"]) % 360 for a, b in moves]
print("on a quarter-turned photo, the job of ... is done by ...:", moves)
print("bearing shifts:", shift)
check("median bearing shift, quarter turn (degrees)", float(np.median(shift)), 87, tol=1)
check("mean correlation with the best match", float(np.mean(Cm.max(1))), 0.78, tol=0.01)
check("mean correlation with the same unit", float(np.mean(np.diag(Cm))), 0.35, tol=0.01)

## 7. The rule in the weights (Plate XII)
The layer below, mixed3a, has its own simpler units. Some answer straight **lines** (we note each one's orientation),
some answer small arcs, **early curves** (we note which way they bulge), and the rest do neither. We sort them by their
answers to the same drawings: a line unit is sharp for lines and likes a line at least 0.8× as much as any small arc; an
early curve is sharp for small arcs and likes one at least 1.5× more than any line. Both must answer strongly: above
2.5× the median unit's best (the strictness the lecture chose on held-out photos).

In [ ]:
def classify_3a(arc_post, line_post, floor_frac=2.5):
    R, A, U = arc_post.shape
    a_rad = np.radians(np.asarray(ANGLES, dtype=float)); l_rad = np.radians(np.arange(line_post.shape[0]) * 2.0)
    small = [i for i, r in enumerate(RADII) if r <= 48]
    floor = floor_frac * float(np.median(np.maximum(arc_post.max((0, 1)), line_post.max(0))))
    out = []
    for u in range(U):
        a = arc_post[small, :, u]; ri = int(a.max(1).argmax()); t = a[ri]; l = line_post[:, u]
        apk, lpk = float(t.max()), float(l.max())
        dsi = abs((t * np.exp(1j * a_rad)).sum()) / (t.sum() + 1e-6)        # 1 = one sharp direction, 0 = flat
        osi = abs((l * np.exp(2j * l_rad)).sum()) / (l.sum() + 1e-6)        # the same for orientation
        d = {"unit": u, "kind": None, "peak": max(apk, lpk)}
        if dsi > 0.4 and apk >= 1.5 * lpk and apk > floor:
            d.update(kind="curve", psi=float(ANGLES[int(t.argmax())]), radius=float(RADII[small[ri]]), peak=apk)
        elif osi > 0.5 and lpk >= 0.8 * apk and lpk > floor:
            d.update(kind="line", theta=float(np.degrees(l_rad[int(l.argmax())])), peak=lpk)
        out.append(d)
    return out

units3a = classify_3a(ARC3A, LIN["mixed3a"])
lines3a = [(d["unit"], round(d["theta"])) for d in units3a if d["kind"] == "line"]
early3a = [(d["unit"], round(d["psi"]), round(d["radius"])) for d in units3a if d["kind"] == "curve"]
print("line units (unit, orientation):", lines3a)
print("early curves (unit, bulge, radius):", early3a)
check("line units", lines3a, [(65, 168), (69, 120), (70, 70), (146, 140), (157, 108), (160, 30), (165, 104), (169, 150)])
check("early curves", early3a, [(71, 210, 32), (81, 60, 32), (83, 108, 32), (92, 352, 48), (95, 282, 24), (96, 40, 32), (100, 166, 24), (104, 30, 24), (133, 234, 32), (137, 94, 24), (145, 320, 32), (163, 226, 16), (173, 184, 32), (189, 108, 32)])

Between mixed3a and a curve detector sit two steps: a 1 × 1 squeeze to 16 maps (the *bottleneck*), then a 5 × 5
filter. Multiply the two and you get the curve detector's weight from every mixed3a unit at each of 5 × 5 places, its
*expanded weights*. Sum each early curve's weights, scale by how strongly that unit fires, and sort by the angle
between the way it bulges and the way the detector's curve bulges.

In [ ]:
mods = dict(net.named_modules())
Wb = mods["mixed3b_5x5_bottleneck_pre_relu_conv"].weight[:, :, 0, 0].cpu()   # 16 x 256
W5 = mods["mixed3b_5x5_pre_relu_conv"].weight.cpu()                         # 96 x 16 x 5 x 5
E = {c: torch.einsum("bhw,bp->phw", W5[c - 320], Wb).numpy() for c in CURVES}  # 256 x 5 x 5 (5x5 branch: 320..415)
curves3a = [d for d in units3a if d["kind"] == "curve"]
xs, ys, bins = [], [], {}
for c in CURVES:
    for d in curves3a:
        dl = ((d["psi"] - ours[c]["phi"] + 180) % 360) - 180
        w = float(E[c][d["unit"]].sum()) * d["peak"]
        xs.append(math.cos(math.radians(dl))); ys.append(w)
        b_ = int(round(dl / 45.0)) * 45; bins.setdefault(180 if b_ == -180 else b_, []).append(w)
mean_by = {b_: float(np.mean(v)) for b_, v in sorted(bins.items())}
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(list(mean_by), list(mean_by.values()), width=30, color=PAPER, edgecolor=INK, hatch="////")
ax.axhline(0, color=INK, lw=0.8); ax.set_xticks(list(mean_by))
ax.set_xlabel("early curve's bulge minus the detector's (degrees)"); ax.set_ylabel("mean weight × its peak")
plt.show()
check("pairs (10 detectors x 14 early curves)", len(xs), 140)
check("same direction: mean weight", mean_by[0], 294.9, tol=0.6)
check("opposite direction: mean weight", mean_by[180], -383.7, tol=0.6)
check("correlation with cos(angle difference)", float(np.corrcoef(xs, ys)[0, 1]), 0.512, tol=0.005)

Early curves bulging the same way excite a curve detector; those bulging the opposite way hold it back.

## 8. Written by hand (Plate XIV)
Can we write a curve detector without learning anything? **Variant A, from pixels**, in two layers:

1. **Edges.** 16 edge detectors, one per orientation (every 11.25°). Each is a pair of *Gabor* filters (a stripe
   pattern under a soft blob, one even and one odd), and its answer is the pair's combined strength, so dark-on-light
   and light-on-dark count the same. Subtract the average over the 16, so plain texture cancels.
2. **The curve.** For every point on the target arc and every edge orientation: +1 if the edge runs along the arc
   there, −1 if it crosses it, cos 2(θ − tangent) in between, spread over a small blob round the point.

The only thing we take from the network is *which* curve each detector should find: its bulge and best radius.

In [ ]:
N_ORIENT = 16

def arc_points(phi_deg, radius, n=64):
    phi = math.radians(phi_deg)
    half = (min(1.6 * radius, 90.0) / radius) / 2
    ox, oy = -radius * math.cos(phi), radius * math.sin(phi)
    ts = np.linspace(-half, half, n)
    xs, ys = ox + radius * np.cos(phi + ts), oy - radius * np.sin(phi + ts)
    tau = (np.degrees(phi + ts) + 90.0) % 180.0                 # the arc's direction there (a line angle)
    return xs, ys, ts, tau, half


def gabor_bank(sigma=2.0, lam=6.0, gamma=0.5, size=15):
    r = size // 2
    Y, X = np.mgrid[-r:r + 1, -r:r + 1].astype(np.float64)
    ev, od = [], []
    for k in range(N_ORIENT):
        th = math.radians(k * 180.0 / N_ORIENT)
        xp, yp = X * math.sin(th) + Y * math.cos(th), X * math.cos(th) - Y * math.sin(th)   # across, along
        env = np.exp(-(xp ** 2 + (gamma * yp) ** 2) / (2 * sigma ** 2))
        e = env * np.cos(2 * math.pi * xp / lam); e -= e.mean()
        o = env * np.sin(2 * math.pi * xp / lam)
        ev.append(e / np.abs(e).sum()); od.append(o / np.abs(o).sum())
    return np.stack(ev + od)[:, None].astype(np.float32)


def curve_kernel_px(phi_deg, radius, half_win=48, blob=1.5):
    xs, ys, ts, tau, half = arc_points(phi_deg, radius, n=96)
    S = 2 * half_win + 1
    Y, X = np.mgrid[-half_win:half_win + 1, -half_win:half_win + 1].astype(np.float64)
    W = np.zeros((N_ORIENT, S, S))
    for x, y, t, ta in zip(xs, ys, ts, tau):
        g = np.exp(-((X - x) ** 2 + (Y - y) ** 2) / (2 * blob ** 2))
        wt = math.cos(0.5 * math.pi * t / half)                 # fade toward the arc's ends
        for k in range(N_ORIENT):
            W[k] += wt * math.cos(2 * math.radians(k * 180.0 / N_ORIENT - ta)) * g
    return (W / np.abs(W).sum()).astype(np.float32)


class PixelCurves:
    # Variant A: a picture [B, 3, 224, 224] -> [B, 10, 28, 28], read out where mixed3b's units look.
    def __init__(self, phis, radii):
        self.gab = torch.tensor(gabor_bank(), device=DEV)
        self.K = torch.tensor(np.stack([curve_kernel_px(p, r) for p, r in zip(phis, radii)]), device=DEV)
        self.off = int(round(CX - 8 * 14))

    def edges(self, img):
        y = (0.299 * img[:, 0] + 0.587 * img[:, 1] + 0.114 * img[:, 2])[:, None]
        r = F.conv2d(F.pad(y, (7, 7, 7, 7), mode="replicate"), self.gab)
        E_ = (r[:, :N_ORIENT] ** 2 + r[:, N_ORIENT:] ** 2 + 1e-8).sqrt()
        return E_ - E_.mean(1, keepdim=True)

    def raw(self, img):
        E_ = self.edges(img)
        lo, hi = 48 - self.off, 48 + (8 * 27 + self.off) - 223
        return F.conv2d(F.pad(E_, (lo, hi, lo, hi)), self.K, stride=8)


phis = [float(ours[c]["phi"]) for c in CURVES]
rads = [float(ours[c]["radius"]) for c in CURVES]
handA = PixelCurves(phis, rads)
fig, axs = plt.subplots(1, 2, figsize=(7, 3.4))
k379 = handA.K[CURVES.index(379)].cpu().numpy()
axs[0].imshow(gabor_bank()[0, 0], cmap="gray"); axs[0].set_title("one edge filter (even)", fontsize=9)
axs[1].imshow(k379.max(0), cmap="gray_r"); axs[1].set_title("3b:379's hand-written curve, its strongest votes", fontsize=9)
for a in axs: a.axis("off")
plt.show()

To put it on the network's scale we fit two things on photos 0–999 (the *fit* photos), never on the photos we test
with: one shared threshold, and one multiplier per detector, so its answers line up with the real one's.

In [ ]:
CUR_IDX = torch.tensor(CURVES, device=DEV)

def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm() + 1e-9))

# the real ten, Variant A's raw answer, and mixed3a (for Variant B, below), on the 1,000 fit photos
nat, rawA, a3 = [], [], []
with torch.no_grad(), Taps(["mixed3a", "mixed3b"]) as T:
    for s in range(0, 1000, 100):
        x = torch.stack([decode(photos[n][0]) for n in range(s, s + 100)]).to(DEV)
        net(prep(x))
        nat.append(T["mixed3b"][:, CURVES].float().cpu()); rawA.append(handA.raw(x).cpu()); a3.append(T["mixed3a"].half().cpu())
natF, rawAF, a3F = torch.cat(nat), torch.cat(rawA), torch.cat(a3)


def choose(raw):
    best = None
    for frac in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
        q = torch.quantile(raw.transpose(0, 1).reshape(10, -1)[:, ::7], 0.999, dim=1)
        b = frac * q
        h = torch.relu(raw - b[None, :, None, None])
        score = float(np.mean([corr(h[:, i], natF[:, i]) for i in range(10)]))
        if best is None or score > best[0]:
            best = (score, frac, b)
    score, frac, b = best
    h = torch.relu(raw - b[None, :, None, None])
    alpha = torch.stack([(h[:, i] * natF[:, i]).sum() / ((h[:, i] ** 2).sum() + 1e-9) for i in range(10)])
    return b.to(DEV), alpha.to(DEV), frac


bA, aA, fracA = choose(rawAF)
print(f"Variant A: threshold {fracA:.1f} x its 99.9th percentile; multipliers {[round(float(a)) for a in aA]}")
check("Variant A's threshold (share of its 99.9th percentile)", fracA, 0.0)

In [ ]:
# Variant A's own tuning curve for 3b:379, on the same arcs as the real one
i379 = CURVES.index(379)
x = torch.stack([arc(a, rads[i379]) for a in ANGLES]).to(DEV)
with torch.no_grad():
    tA = (torch.relu(handA.raw(x) - bA[None, :, None, None]) * aA[None, :, None, None])[:, i379, 14, 14].cpu().numpy()
real = ARC3B[RADII.index(int(rads[i379])), :, 379]
fig, ax = plt.subplots(figsize=(8, 3))
B_ = [bearing(a) for a in ANGLES]
ax.plot(B_, real, ".", ms=3, color=INK, label="3b:379, the real neuron")
ax.plot(B_, tA, ".", ms=3, color=INK2, label="written by hand (A)")
ax.set_xlabel("bearing (degrees)"); ax.set_xticks(range(0, 361, 45)); ax.legend(); plt.show()
pkA = int(np.argmax(tA))
check("A's preferred bearing for 3b:379", bearing(ANGLES[pkA]), bearing(114))
check("A against the real tuning curve, correlation", float(np.corrcoef(real, tA)[0, 1]), 0.687, tol=0.01)
check("A's echo: its answer at the opposite direction, share of peak", float(tA[(pkA + 90) % 180] / tA[pkA]), 0.355, tol=0.01)

The hand-written detector finds the right curve, but it also answers a curve bulging the opposite way, at about a third
of its peak (an *echo*). The real neuron says nothing there: it has learned to count the opposite curve against itself.
That is exactly what the weights in section 7 showed.

**Variant B, from the network's own parts:** the same idea one layer up. Its inputs are mixed3a's line units and early
curves from section 7, not pixels. Each line unit votes with cos 2(θ − tangent) near each point of the arc, and each early
curve votes with cos(its bulge − the arc's bulge there). The votes are divided by the unit's peak and laid on a 5 × 5
grid, like the real filter.

In [ ]:
def curve_kernel_3a(phi_deg, radius, units, A=1.0, B=1.0, blob=6.0, k=5):
    xs, ys, ts, tau, half = arc_points(phi_deg, radius, n=64)
    r = k // 2
    W = np.zeros((len(units), k, k))
    for d in units:
        if d["kind"] is None:
            continue
        for gy in range(-r, r + 1):
            for gx in range(-r, r + 1):
                g = np.exp(-((xs - 8 * gx) ** 2 + (ys - 8 * gy) ** 2) / (2 * blob ** 2))
                wt = np.cos(0.5 * np.pi * ts / half)
                if d["kind"] == "line":
                    c_ = np.cos(2 * np.radians(d["theta"] - tau)); W[d["unit"], gy + r, gx + r] = A * (g * wt * c_).sum() / d["peak"]
                else:
                    c_ = np.cos(np.radians(d["psi"]) - (np.radians(phi_deg) + ts)); W[d["unit"], gy + r, gx + r] = B * (g * wt * c_).sum() / d["peak"]
    return W.astype(np.float32)


def kernB(A=1.0, B=1.0):
    return torch.tensor(np.stack([curve_kernel_3a(p, r, units3a, A, B) for p, r in zip(phis, rads)]), device=DEV)


def rawB(a3_, W):
    return torch.cat([F.conv2d(a3_[i:i + 200].to(DEV).float(), W, padding=2).cpu() for i in range(0, len(a3_), 200)])


WB = kernB()
bB, aB, fracB = choose(rawB(a3F, WB))
print(f"Variant B: threshold {fracB:.1f} x its 99.9th percentile")
check("Variant B's threshold", fracB, 0.2)
with torch.no_grad(), Taps(["mixed3a"]) as T:
    net(prep(x))
    tB = (torch.relu(F.conv2d(T["mixed3a"], WB, padding=2) - bB[None, :, None, None]) * aB[None, :, None, None])[:, i379, 14, 14].cpu().numpy()
check("B against the real tuning curve for 3b:379, correlation", float(np.corrcoef(real, tB)[0, 1]), 0.761, tol=0.01)

## 9. Does it still work? (Plates XVI–XVII)
Now the real test. Take the ten curve detectors out of the network and put something else in their place, then score
the whole network on photos 1,000–2,999 (photos it was never tuned on here). *Top-5* means the right class is among
its five best guesses. The conditions:
- **as trained**;
- **removed**: the ten set to zero;
- **replaced by their average value**;
- **Variant A** and **Variant B** in their place;
- **ten random other neurons** of the same 5 × 5 branch removed instead, for scale.

In [ ]:
mean_val = natF.mean((0, 2, 3)).to(DEV)
RAND = [[321, 323, 327, 336, 346, 350, 366, 375, 394, 398], [320, 322, 346, 358, 372, 380, 393, 394, 400, 401], [320, 322, 332, 357, 358, 363, 370, 382, 383, 412]]                     # the lecture's three random sets, drawn with numpy seed 0
state = {"mode": "base", "x": None, "a3": None}

def edit3b(o):
    md_ = state["mode"]
    if md_ == "base":
        return o
    o = o.clone()
    if md_ == "zero":
        o[:, CUR_IDX] = 0
    elif md_ == "mean":
        o[:, CUR_IDX] = mean_val[None, :, None, None].expand(o.shape[0], -1, o.shape[2], o.shape[3])
    elif md_ == "A":
        o[:, CUR_IDX] = torch.relu(handA.raw(state["x"]) - bA[None, :, None, None]) * aA[None, :, None, None]
    elif md_ == "B":
        o[:, CUR_IDX] = torch.relu(F.conv2d(state["a3"], WB, padding=2) - bB[None, :, None, None]) * aB[None, :, None, None]
    elif md_.startswith("rand"):
        o[:, torch.tensor(RAND[int(md_[-1])], device=DEV)] = 0
    return o


def keep3a(o):
    state["a3"] = o
    return o


def score(conds, first=1000, last=3000):
    hits = {c: 0 for c in conds}
    with torch.no_grad(), Taps(["softmax2_pre_activation_matmul"], edit={"mixed3a": keep3a, "mixed3b": edit3b}) as T:
        for s in range(first, last, 100):
            x = torch.stack([decode(photos[n][0]) for n in range(s, s + 100)]).to(DEV)
            y = torch.tensor(LABELS[s:s + 100], device=DEV)
            state["x"] = x
            for c in conds:
                state["mode"] = c
                net(prep(x))
                top5 = T["softmax2_pre_activation_matmul"][:, PERM].topk(5, 1).indices
                hits[c] += int((top5 == y[:, None]).any(1).sum())
    state["mode"] = "base"
    return {c: 100 * h / (last - first) for c, h in hits.items()}


t = time.time()
TOP5 = score(["base", "zero", "mean", "A", "B", "rand0", "rand1", "rand2"])
print(f"scored in {time.time() - t:.0f} s")
for c, v in TOP5.items():
    print(f"  {c:6s} top-5 {v:.2f}%")
won = lambda c: 100 * (TOP5[c] - TOP5["zero"]) / (TOP5["base"] - TOP5["zero"])
check("as trained, top-5 on photos 1,000-2,999 (%)", TOP5["base"], 87.7, tol=0.11)
check("the ten removed (%)", TOP5["zero"], 84.85, tol=0.11)
check("replaced by their average (%)", TOP5["mean"], 85.15, tol=0.11)
check("replaced by Variant A, from pixels (%)", TOP5["A"], 86.05, tol=0.11)
check("replaced by Variant B, from mixed3a (%)", TOP5["B"], 86.5, tol=0.11)
for k in range(3):
    check(f"ten random 5x5 neurons removed instead, set {k + 1} (%)", TOP5[f"rand{k}"], [87.1, 87.2, 87.1][k], tol=0.11)
check("share of the loss Variant A wins back (%)", won("A"), 42.1, tol=4)
check("share of the loss Variant B wins back (%)", won("B"), 57.9, tol=4)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3))
names = ["as trained", "removed", "average", "A: pixels", "B: mixed3a", "random 10"]
vals = [TOP5["base"], TOP5["zero"], TOP5["mean"], TOP5["A"], TOP5["B"], np.mean([TOP5[f"rand{k}"] for k in range(3)])]
ax.bar(names, vals, color=PAPER, edgecolor=INK, hatch=["", "", "..", "////", "xxxx", ""])
ax.set_ylim(83, 88.5); ax.set_ylabel("top-5 (%)"); ax.set_title("Photos 1,000–2,999", fontsize=10)
for i, v in enumerate(vals):
    ax.text(i, v + 0.08, f"{v:.2f}", ha="center", fontsize=9)
plt.show()

Hand-written detectors win back a real share of what removing the curve detectors costs: from pixels about two-fifths
here, and from the network's own lines and early curves more than half. (On all 49,000 test photos the lecture measured
a third and two-thirds; the paper reports 57% on 2,000.) Removing ten *random* neurons costs far less: the curve
detectors matter more than their number suggests.

## Your turn

1. **The tightest detector.** Find 3b:406's preferred bearing and width from `ARC3B`. *Hint: copy section 5's loop for
   one unit.*
2. **A curve you weren't told about.** 3b:380 is not one of the ten. Measure its bearing, best radius, and peak as a share
   of the median peak of the ten. *Hint: `describe()` on `ARC3B[:, :, 380]` at its best radius.*
3. **Change the pen.** Draw 3b:379's test arcs with a pen twice as thick (`width=4.0`), then white on black
   (`bg=0.0, ink=1.0`). Does its preferred bearing move? *Hint: `respond([arc(a, 64, width=4.0) for a in ANGLES])`.*
4. **Early curves alone.** Build Variant B from early curves only (`kernB(A=0.0, B=1.0)`), fit its threshold with
   `choose()`, and score it. *Hint: add a mode to `edit3b`, or swap `WB`, `bB`, `aB` for a moment.*
5. **Half a turn.** Turn the photos 180° instead of 90°. How far round does each job move? *Hint: `peaks_turned(2)`.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

In [ ]:
# Your turn 5

## Solutions

In [ ]:
# 1. The tightest detector
rb = int(np.argmax(ARC3B[:, :, 406].max(1)))
pk, peak, width, opp = describe(ARC3B[rb, :, 406])
print(f"3b:406: bearing {bearing(ANGLES[pk])}°, width {width}°")
check("3b:406's bearing", bearing(ANGLES[pk]), 356)
check("3b:406's width", width, 22)

In [ ]:
# 2. A curve you weren't told about
rb = int(np.argmax(ARC3B[:, :, 380].max(1)))
pk, peak, width, opp = describe(ARC3B[rb, :, 380])
med = float(np.median([ARC3B[:, :, c].max() for c in CURVES]))
print(f"3b:380: bearing {bearing(ANGLES[pk])}°, best radius {RADII[rb]} px, peak {peak / med:.2f} x the ten's median, opposite {opp:.2f}")
check("3b:380's bearing", bearing(ANGLES[pk]), 248)
check("3b:380's best radius", RADII[rb], 64)
check("3b:380's peak over the ten's median", peak / med, 1.19, tol=0.01)

In [ ]:
# 3. Change the pen
for name, kw, want_b, want_p in (("thick pen", dict(width=4.0), 334, 1009.0), ("white on black", dict(bg=0.0, ink=1.0), 334, 980.3)):
    t_ = respond([arc(a, 64, **kw) for a in ANGLES])["mixed3b"][:, 379]
    pk = int(np.argmax(t_))
    print(f"{name}: bearing {bearing(ANGLES[pk])}°, peak {t_[pk]:.1f}")
    check(f"3b:379, {name}: bearing", bearing(ANGLES[pk]), want_b)
    check(f"3b:379, {name}: peak", float(t_[pk]), want_p, tol=0.006 * want_p)
# It barely moves (2 degrees): the neuron answers the curve, not the pen.

In [ ]:
# 4. Early curves alone
WB_all, bB_all, aB_all = WB, bB, aB
WB = kernB(A=0.0, B=1.0)
bB, aB, _ = choose(rawB(a3F, WB))
only = score(["B"])["B"]
WB, bB, aB = WB_all, bB_all, aB_all
print(f"early curves only: top-5 {only:.2f}%")
check("Variant B, early curves only (%)", only, 86.55, tol=0.11)
# Almost all of Variant B's win comes from the early curves.

In [ ]:
# 5. Half a turn
P2 = peaks_turned(2)
C2 = np.corrcoef(P2.T, P0.T)[:10, 10:]
shift2 = [(ours[CURVES[int(np.argmax(C2[i]))]]["bearing"] - ours[CURVES[i]]["bearing"]) % 360 for i in range(10)]
print("bearing shifts, half a turn:", shift2)
check("median bearing shift, half turn (degrees)", float(np.median(shift2)), 190, tol=1)
print(f"whole notebook: {(time.time() - T0) / 60:.1f} minutes")